This notebook does step 1-3 (without LLM topic extraction) for LU and Delft: 
1. Choosing universities to analyze: We browse the QS index page and select 5 randomly for the 'computer science information systems' category, per continent (Asia, Europe, North America, Latin America, Africa, Oceania).
For now, we focus on Europe + the universities that pop up in the test dataset.
2. Choose (manually) programs at the universities that:
   - fall under the computer science umbrella (data science, computer science, information science, software science/engineering) 
   - are BSc or MSc programs (not BEng or PhD)
   - have English course descriptions
    Find websites for all of these selected programs.
3. Per program, extract the following info for courses included in the program (+ suggested electives?)
   Also do this for the test dataset.
   { 

  "title": "...", 

  "credits": "...", 

  "level": "BSc / MSc / ...", 

“description: “,  

  "topics": ["...", "..."],           // extracted or inferred (LLM / SWEBOK?) 

  "learning_outcomes": ["...", "..."], // explicit or inferred from exam/content description 

  "prerequisites": ["...", "..."], 

  "examination_methods": ["...", "..."], 

}  

4. Find similarity between courses in a hierarchical way (start with description, topics and/or LOs. Then look at prerequisites of one course vs LOs/descriptions of another course, level and examination methods. 

5. Do this for the test dataset and see whether the similarity matches what we would expect. Analyze this per data source (LO, description, topic). Is there a difference between:

- embedding model (sentenceBERT vs TF-IDF vs bigger document-level embedding) 
- data source (only LO matching, description, topic)
- topic extraction method (LLM / SWEBOK)
Other idea for test case is Dutch TUs. 

In [1]:
!pip install playwright 
import playwright

### 

### start with Delft
TU Delft 
Research in Cyber Security - Hacking Lab  
https://studiegids.tudelft.nl/courses/study-guide/educations/14429  

Smart Phone Sensing 
https://studiegids.tudelft.nl/courses/study-guide/educations/14704  

Distributed Data Systems 
https://studiegids.tudelft.nl/courses/study-guide/educations/34135  
EDAP25 

Systems Security 
https://studiegids.tudelft.nl/courses/study-guide/educations/14761  

Error-Correcting Codes  
https://studiegids.tudelft.nl/courses/study-guide/educations/14458  
EITN70  
Saved URL info by right-clicking after accessing the URL. 

### LU

In [56]:
### Try scraping LU webpages 
async def scrape_lth_courses_direct(url):
    """
    Scrape courses by waiting for the specific table structure
    """
    async with async_playwright() as p:
        browser = await p.firefox.launch(headless=True)
        page = await browser.new_page()
        
        # Navigate with specific wait conditions
        await page.goto(url)
        
        # Wait for any content to load
        await page.wait_for_timeout(5000)
        
        # Get all text content
        content = await page.content()
        
        # Look for course codes in the page source
        course_codes = await page.evaluate('''
            () => {
                const text = document.body.innerText;
                const codePattern = /[A-Z]{2,5}\\d{2,5}/g;
                const matches = text.match(codePattern) || [];
                return [...new Set(matches)];  // Remove duplicates
            }
        ''')
        
        # Build course URLs from codes
        courses = []
        for code in course_codes:
            if len(code) >= 5 and len(code) <= 8:  # Filter valid course codes
                courses.append({
                    'course_code': code,
                    'url': f'https://kurser.lth.se/lot/course/{code}',
                    'url_kp': f'https://kurser.lth.se/lot/course-syllabus-en/26_27/{code}',
                    'programme': 'C',
                    'academic_year': '26_27'
                })
        
        await browser.close()
        
        # Create DataFrame
        df = pd.DataFrame(courses)
        return df
        
url = 'https://kurser.lth.se/lot/programme?programme=C&ay=26_27#EITA55'
df = await scrape_lth_courses_direct(url)
print(df)

    course_code                                      url  \
0        FMAB65  https://kurser.lth.se/lot/course/FMAB65   
1        EDAB05  https://kurser.lth.se/lot/course/EDAB05   
2        FMAB70  https://kurser.lth.se/lot/course/FMAB70   
3        EITA66  https://kurser.lth.se/lot/course/EITA66   
4        EDAA01  https://kurser.lth.se/lot/course/EDAA01   
..          ...                                      ...   
170      EITL01  https://kurser.lth.se/lot/course/EITL01   
171      MAML15  https://kurser.lth.se/lot/course/MAML15   
172      EXTL03  https://kurser.lth.se/lot/course/EXTL03   
173      FMAL01  https://kurser.lth.se/lot/course/FMAL01   
174      FMSL01  https://kurser.lth.se/lot/course/FMSL01   

                                                url_kp programme academic_year  
0    https://kurser.lth.se/lot/course-syllabus-en/2...         C         26_27  
1    https://kurser.lth.se/lot/course-syllabus-en/2...         C         26_27  
2    https://kurser.lth.se/lot/cours

In [30]:
# split into mandatory and elective manually 
# MAMF50 is the first elective course for the C program  
#print([i for i in range(len(df)) if df['course_code'][i] == 'MAMF50'])
# so 0-29 are mandatory 
# FMAN20 is the first elective course for the D program
#print([i for i in range(len(df)) if df['course_code'][i] == 'FMAN20'])
# so 0-31 are mandatory 

[32]


In [57]:
# now, for each of the LU urls, we want to extract the data. 

# Step 1: Just save the HTML (no parsing yet)
async def save_html_only(url, output_path):
    async with async_playwright() as p:
        browser = await p.firefox.launch(headless=True)
        page = await browser.new_page()
        await page.goto(url, wait_until="domcontentloaded")
        await page.wait_for_timeout(3000)
        
        html = await page.content()
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(html)
        
        await browser.close()
        print(f"Saved: {output_path}")

# Save multiple pages
output_path = '/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/'
urls_and_files = [
    (df['url_kp'][i], output_path+df['course_code'][i]+".html") for i in range(len(df))
    # Add more URLs
]

for url, filename in urls_and_files:
    print(url)
    await save_html_only(url, filename)

https://kurser.lth.se/lot/course-syllabus-en/26_27/FMAB65
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/FMAB65.html
https://kurser.lth.se/lot/course-syllabus-en/26_27/EDAB05
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/EDAB05.html
https://kurser.lth.se/lot/course-syllabus-en/26_27/FMAB70
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/FMAB70.html
https://kurser.lth.se/lot/course-syllabus-en/26_27/EITA66
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/EITA66.html
https://kurser.lth.se/lot/course-syllabus-en/26_27/EDAA01
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/EDAA01.html
https://kurser.lth.se/lot/course-syllabus-en/26_27/MAMA20
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/MAMA20.html
https://kurser.lth.se/lot/course-syllabus-en/26_27/EDAA75
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/EDAA

In [36]:
# ovrigt
output_path = '/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/Övrigt/'
urls_and_files = [
    ('https://kurser.lth.se/lot/course-syllabus-en/26_27/EDAP25', output_path+"EDAP25"+".html"),
    ('https://kurser.lth.se/lot/course-syllabus-en/23_24/EITN70', output_path+"EITN70"+".html"),
    # Add more URLs
]

for url, filename in urls_and_files:
    print(url)
    await save_html_only(url, filename)

https://kurser.lth.se/lot/course-syllabus-en/26_27/EDAP25
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/Övrigt/EDAP25.html
https://kurser.lth.se/lot/course-syllabus-en/23_24/EITN70
Saved: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/Övrigt/EITN70.html


In [38]:
### Now try to parse the HTML files as JSON using their structure 
from bs4 import BeautifulSoup
import json

def parse_html_auto(html_file):
    """Extract structured data from ANY HTML file automatically"""
    with open(html_file, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')
    
    # Remove non-content elements
    for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
        tag.decompose()
    
    result = {
        'title': soup.title.string.strip() if soup.title else '',
        'headings': [h.get_text().strip() for h in soup.find_all(['h1', 'h2', 'h3']) if h.get_text().strip()],
        'paragraphs': [p.get_text().strip() for p in soup.find_all('p') if p.get_text().strip()],
        'lists': [],
        'tables': [],
        'full_text': soup.get_text()
    }
    
    # Extract list items
    for ul in soup.find_all(['ul', 'ol']):
        items = [li.get_text().strip() for li in ul.find_all('li') if li.get_text().strip()]
        if items:
            result['lists'].append(items)
    
    # Extract tables
    for table in soup.find_all('table'):
        rows = []
        for tr in table.find_all('tr'):
            cells = [td.get_text().strip() for td in tr.find_all(['td', 'th']) if td.get_text().strip()]
            if cells:
                rows.append(cells)
        if rows:
            result['tables'].append(rows)
    
    return result

# Use it
html_file = '/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/Övrigt/'+"EDAP25"+".html"
data = parse_html_auto(html_file)
print(json.dumps(data, indent=2))
html_file = '/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/Delft/Studiegids — Study Guide _ Error Correcting Codes.html'
data2 = parse_html_auto(html_file)
print(json.dumps(data2, indent=2))

{
  "title": "Kursplan (en)",
  "headings": [
    "Kursplan",
    "Distributed Systems\nDistribuerade system",
    "EDAP25, 7.5 credits, A (Second Cycle)",
    "General Information",
    "Aim",
    "Learning outcomes",
    "Contents",
    "Examination details",
    "Admission",
    "Reading list",
    "Contact"
  ],
  "paragraphs": [
    "Valid for: 2026/27\nFaculty: Faculty of Engineering LTH\nDecided by: PLED C/D\nDate of Decision: 2026-04-16",
    "Depth of study relative to the degree requirements:  Second cycle, in-depth level of the course cannot be classified\n      \n      \nElective for: C4, D4, E4, F4, Pi4\nLanguage of instruction: The course will be given in English",
    "To give an introduction to the fundamental concepts of distributed systems, their properties and application in practice.",
    "Knowledge and understanding\n      For a passing grade the student must",
    "Competences and skills\n      For a passing grade the student must",
    "Judgement and approach\n 

In [58]:
# now we're ready to save it and process it using an LLM ! 
import os
uni_dir = '/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/' 
all_files = os.listdir(uni_dir)
html_files = [f for f in all_files if f.lower().endswith('.html')] 
for html_file in html_files: 
    # Create JSON filename by replacing .html with .json
    json_file = uni_dir + html_file.replace('.html', '.json')
    data = parse_html_auto(uni_dir + html_file)
    # Save to JSON file
    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    
    print(f"Saved to: {json_file}")

Saved to: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/MAMN80.json
Saved to: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/EDAL01.json
Saved to: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/ETSN11.json
Saved to: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/FMAF01.json
Saved to: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/EDAF05.json
Saved to: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/IYP000.json
Saved to: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/FRTN01.json
Saved to: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/BMEN20.json
Saved to: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/MAMM20.json
Saved to: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/MIOA15.json
Saved to: /mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/LU/C/FMSN35.json
Saved to: /mimer/NOBA

TU Delft, NL, 2020, <student name>
IN4253ET "Hacking Lab"-Applied Security Analysis 5.0, A, inom sec (nivå beroende på projekt, här implementerades log-jam attack mot SSH)
IN4254 Smart Phone Sensing 5.0, A, valfri, inom kom (inget överlapp med av, interaktionsdesign, fokus på behandling av sensordata och matematiska modeller för det, applicering: lokalisering av användare i wi-fi nätverk)
IN4391 Distributed Systems 5.0, A, räknas som EDAP25 Distribuerade system (inom ks)
CS4150 Systems Security 5.0, A, inom sec
CS4210-A Algorithms for Intelligent Decision Making 5.0, A, inom pv
ET4030 Error-Correcting Codes 4.0, A, räknas som EITN70 Kanalkodning för tillförlitlig kommunikation (inom ks)

University of California, Santa Cruz, 2020, <student name>
CSE 80A, Universal Access, G, räknas som TNSF05 Rehabiliteringsteknik
CMPM 290K, Social and Emotional Approaches to Human Computer Interaction, A inom da
CMPM 26, Introduction to 3D Animation, G, överlappar 3 hp med MAMF45 Virtual reality i teori och praktik, inom da
CMPM 203, Computational Media Methods, A inom da
CMPM 177, Creative Strategies for Designing Interactive Media, A inom da
CSE 160, Intro Comp Graphics, G, räknas som EDAF80 Datorgrafik
CSE 163, Data Programming for Visualization, A valfri
SPAN 1, Spanish: First Year, externt valfri
CSE 185E, Technical Writing for Computer Engineers, externt valfri, blockar GEMA20 Engelska för tekniker

University of California, Santa Cruz, 2019, <student name>, <student name>
CMPS160, Intro to Computer Graphics, räknas som EDAF80 Datorgrafik
CMPS140 Artificial Intelligence, räknas som EDAP01 Artificiell intelligens
CMPS144, Applied Machine Learning, räknas som EDAN95 Tillämpad maskininlärning
CMPS130, Computational Models, inget överlapp (kollat med Per), A valfri, inom pv
CMPS111, Introduction to Operating Systems, överlappar 2,5 hp med EDAF35 Operativsystem (G2), resten överlappar med EDAP10 Flertrådad programmering (A) [CMPS111 räknas som A-nivå] No exact current offering on website. 
CMPS121, Mobile Applications, G valfri
143E, History of Design, externt valfri
2, First Year Italian, externt valfri

TU München, 2019, <student name>
https://campus.tum.de/tumonline/ee/ui/ca2/app/desktop/#/slc.tm.cp/student/courses?$ctx=design=ca;lang=en&$skip=0&objTermId=193&orgId=1&q=javascript 
IN2026, Visual Data Analytics, externt valfri
IN2060, Real-time Systems, räknas som EDAP10 Flertrådad programmering (A)
IN2062, Techniques in Artificial Intelligence, räknas som EDAP01 Artificiell intelligens (A)
IN2073, Cloud Computing, A valfri, inom pv, sec & ks
IN2081, Patterns in Software Engineering, överlapp 2 hp med OMD, G2 valfri
Ingen kurskod, Advanced Seminar Course JavaScript Technology, A valfri, inom pv couldn't find it 

TU München, 2019, <student name>
IN2026, Visual Data Analytics, externt valfri
IN2028, Business Analytics, G2 valfri
IN2081, Patterns in Software Engineering, överlapp 2 hp med OMD, G2 valfri
IN2106, Hands-on Deep Learning for Computer Vision and Biomedicine, räknas som EDAN70 Projekt i datavetenskap (A)